# 30 · Triage student v2: train variants A and B once

Runs step 5 of [`docs/experiments/triage_student_v2/PROTOCOL.md`](../docs/experiments/triage_student_v2/PROTOCOL.md)
(version 1), locked before any student existed, in about **25–40 minutes**. Use a **T4 GPU runtime**
(Runtime → Change runtime type → T4 GPU).

It checks out the pinned code commit, fetches the Booking CSV and checks its SHA-256, rebuilds every review text
from it and stops unless each matches the committed keys, then trains DistilBERT twice on the teacher's labels of
the 27,052 training reviews with the first protocol's settings: **A** without class weights, **B** with
`pos_weight = min(negatives / positives, 20)`. It writes each variant's logits for the development, test and
held-out reviews.

Nothing is scored here: thresholds, the choice of variant and the one test score are later, local steps. No API
key and no Hugging Face token are needed.

**Weights.** With `KEEP_WEIGHTS = True` (the default) both models are saved to your Google Drive, in
`MyDrive/hotel-review-nlp/triage_student_v2/A` and `/B`, never in the repository; Colab asks once for access to
Drive. They are needed later for CPU latency and clause evidence. Set it to `False` to keep none.

**Send back** `triage_student_v2_results.zip`. It holds the logits and `run.json`, not the weights.

In [ ]:
from pathlib import Path

SOURCE_COMMIT = "fa0ed372b4934f50a4f864808f3af1a230033a3c"
REPO = Path("/content/hotel-review-nlp")
OUT = Path("docs/experiments/triage_student_v2/student")  # inside the repository: logits and run.json only
KEEP_WEIGHTS = True
WEIGHTS = Path("/content/drive/MyDrive/hotel-review-nlp/triage_student_v2")  # outside the repository

In [ ]:
import os
import subprocess
import sys

if not REPO.exists():
    subprocess.run(["git", "clone", "--quiet", "https://github.com/krimits/hotel-review-nlp.git", str(REPO)], check=True)
os.chdir(REPO)
subprocess.run(["git", "fetch", "--quiet", "origin"], check=True)
subprocess.run(["git", "checkout", "--quiet", "--detach", SOURCE_COMMIT], check=True)
GIT_COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
if GIT_COMMIT != SOURCE_COMMIT:
    raise RuntimeError(f"checked out {GIT_COMMIT}, not the pinned {SOURCE_COMMIT}")
sys.path.insert(0, str(REPO / "src"))
# The transformers version of the project's other runs and of the CI.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "transformers==4.56.2"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "-e", "."], check=True)
print("commit", GIT_COMMIT)

## 1. Data: the Booking CSV the samples were drawn from

In [ ]:
import torch


def run(command: list[str]) -> None:
    print("$", " ".join(command), flush=True)
    subprocess.run(command, check=True)


if not torch.cuda.is_available():
    raise RuntimeError("No GPU: choose Runtime -> Change runtime type -> T4 GPU, then run all again.")
run([sys.executable, "scripts/fetch_booking_515k.py"])  # refuses a file that is not the Kaggle download
print("GPU", torch.cuda.get_device_name(0))

## 2. Google Drive for the weights (skipped when `KEEP_WEIGHTS = False`)

In [ ]:
if KEEP_WEIGHTS:
    from google.colab import drive

    drive.mount("/content/drive")
    if any((WEIGHTS / variant).exists() for variant in ("A", "B")):
        raise RuntimeError(f"{WEIGHTS} already holds a variant; weights from an earlier run are never overwritten")
    WEIGHTS.mkdir(parents=True, exist_ok=True)
    print("weights go to", WEIGHTS)
else:
    print("weights are not kept")

## 3. Train A and B once (about 15–25 minutes)

The script rebuilds every text and stops unless all 31,100 match their keys; then it trains A, writes its logits,
trains B from the same seed, and writes its logits. It refuses to run twice into the same folder.

In [ ]:
command = [sys.executable, "scripts/train_triage_student_v2.py", "--csv", "data/raw/booking_reviews_515k.csv"]
run(command + (["--weights", str(WEIGHTS)] if KEEP_WEIGHTS else []))

## 4. Collect and download

In [ ]:
import hashlib
import json
import zipfile

collected = sorted(path for path in OUT.iterdir() if path.is_file() and path.name != "artifact_manifest.json")
if any(path.suffix in (".safetensors", ".bin", ".pt") for path in collected):
    raise RuntimeError("model weights in the bundle")
names = {path.name for path in collected}
expected = {"run.json", *(f"logits_{v}_{s}.csv" for v in ("A", "B") for s in ("development", "test", "held_out"))}
if names != expected:
    raise RuntimeError(f"the bundle holds {sorted(names)}, not {sorted(expected)}")
manifest = {"commit": GIT_COMMIT,
            "files": [{"path": path.as_posix(), "bytes": path.stat().st_size,
                       "sha256": hashlib.sha256(path.read_bytes()).hexdigest()} for path in collected]}
(OUT / "artifact_manifest.json").write_text(json.dumps(manifest, indent=1) + "\n", encoding="utf-8")
with zipfile.ZipFile("triage_student_v2_results.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for path in [*collected, OUT / "artifact_manifest.json"]:
        archive.write(path, path.as_posix())
record = json.loads((OUT / "run.json").read_text(encoding="utf-8"))
print({v: r["loss_per_epoch"] for v, r in record["variants"].items()}, "weights kept:", record["weights_kept"],
      f"{len(collected) + 1} files")

In [ ]:
from google.colab import files

files.download("triage_student_v2_results.zip")

## 5. After the run

Send `triage_student_v2_results.zip` back. Its paths are the repository's: the files are checked against
`artifact_manifest.json` and the pinned commit, and committed. Then, locally: the thresholds of A, B and C are
chosen on the development set, one variant is chosen by the protocol's rule, and all of it is committed; only
after that is the chosen variant scored on the test set, once.